In [1]:
import os
print(os.getcwd())

C:\Users\DELL


In [2]:
import os
print(os.getcwd())

C:\Users\DELL\loan-amount-prediction-linear-regression


In [3]:
import pandas as pd
import numpy as np

In [5]:
df = pd.read_csv(r"C:\Users\DELL\Downloads\loan_amount_regression_messy.csv")

In [7]:
print("Shape:", df.shape)
display(df.head(10))
display(df.dtypes)
display(df.isna().sum())
print("Unique applications:",
df["applicant_id"].nunique())
print("Repeated applicant rows:",
df.duplicated("applicant_id").sum())

Shape: (2090, 11)


,applicant_id,age,monthly_income,credit_score,existing_loans,employment_type,loan_tenure_months,requested_amount,current_monthly_emi,approved_loan_amount,record_updated_at
0,APP00924,27,106288,626,2,Business,48,1876000,48560,160000,2025-06-15
1,APP01908,28,68754,637,0,Salaried,36,1160000,15831,316000,2025-05-08
2,APP01221,47,51912,681,3,Salaried,24,217000,4650,217000,2025-09-28
3,APP01931,25,67500,787,3,Salaried,24,902000,8626,416000,2025-06-18
4,APP00680,58,61386,798,4,Salaried,60,667000,23859,535000,2025-04-17
5,APP00136,33,86625,756,0,SALARIED,60,1327000,22288,747000,2025-03-30
6,APP01539,49,101060,809,3,Business,24,743000,4190,743000,2025-01-30
7,APP01481,27,52170,706,1,Business,12,578000,3197,337000,2025-08-28
8,APP00109,40,72475,649,2,self-employed,36,405000,12143,405000,2025-09-17
9,APP01985,46,93384,651,0,Salaried,24,919000,35222,493000,2025-02-04


applicant_id            object
age                      int64
monthly_income          object
credit_score            object
existing_loans           int64
employment_type         object
loan_tenure_months       int64
requested_amount        object
current_monthly_emi     object
approved_loan_amount    object
record_updated_at       object
dtype: object

applicant_id             0
age                      0
monthly_income          58
credit_score            43
existing_loans           0
employment_type         58
loan_tenure_months       0
requested_amount         0
current_monthly_emi     48
approved_loan_amount    24
record_updated_at        0
dtype: int64

Unique applications: 2000
Repeated applicant rows: 90


In [8]:
df.dtypes

applicant_id            object
age                      int64
monthly_income          object
credit_score            object
existing_loans           int64
employment_type         object
loan_tenure_months       int64
requested_amount        object
current_monthly_emi     object
approved_loan_amount    object
record_updated_at       object
dtype: object

In [9]:
df["monthly_income"].dropna().sample(10, random_state=42).tolist()

['52286',
 '40175',
 '65225',
 '131845',
 '74291',
 '63439',
 '73303',
 '164450',
 '70910',
 '59658']

In [10]:
df.loc[
    df["monthly_income"].astype("string").str.contains("₹", regex=False, na=False),
    "monthly_income"
].head(10)

19     ₹187,451
37      ₹32,070
70      ₹52,549
87      ₹93,044
100     ₹41,366
162     ₹55,499
187     ₹65,460
194     ₹53,274
198     ₹82,017
210     ₹65,890
Name: monthly_income, dtype: object

In [11]:
df["record_updated_at"] = pd.to_datetime(df["record_updated_at"])

df = (
    df.sort_values("record_updated_at")
    .drop_duplicates(subset="applicant_id", keep="last")
    .reset_index(drop=True)
)

print("Rows after duplication:", len(df))
print("Unique applicants:", df["applicant_id"].nunique())
print("Repeated IDs:", df.duplicated("applicant_id").sum())

Rows after duplication: 2000
Unique applicants: 2000
Repeated IDs: 0


In [12]:
amount_cols = [
    "monthly_income",
    "requested_amount",
    "current_monthly_emi",
    "approved_loan_amount"
]

for col in amount_cols:
    cleaned = (
        df[col].astype("string")
               .str.replace("₹", "", regex=False)
               .str.replace(",", "", regex=False)
               .str.strip()
    )
    df[col] = pd.to_numeric(cleaned, errors="coerce")

print(df[amount_cols].dtypes)
print(df[amount_cols].isna().sum())

monthly_income          Int64
requested_amount        Int64
current_monthly_emi     Int64
approved_loan_amount    Int64
dtype: object
monthly_income          57
requested_amount         0
current_monthly_emi     47
approved_loan_amount    24
dtype: int64


In [13]:
pd.to_numeric(pd.Series(["50000", "unknown"]), errors="coerce")

0    50000.0
1        NaN
dtype: float64

In [14]:
amount_cols = [
    "monthly_income",
    "requested_amount",
    "current_monthly_emi",
    "approved_loan_amount"
]

for col in amount_cols:
    cleaned = (
        df[col].astype("string")
               .str.replace("₹", "", regex=False)
               .str.replace(",", "", regex=False)
               .str.strip()
    )
    df[col] = pd.to_numeric(cleaned, errors="coerce")

print(df[amount_cols].dtypes)
print(df[amount_cols].isna().sum())

monthly_income          Int64
requested_amount        Int64
current_monthly_emi     Int64
approved_loan_amount    Int64
dtype: object
monthly_income          57
requested_amount         0
current_monthly_emi     47
approved_loan_amount    24
dtype: int64


In [15]:
df["credit_score"] = pd.to_numeric(df["credit_score"], errors="coerce")

print("dtype:", df["credit_score"].dtype)
print("missing:", df["credit_score"].isna().sum())
print("minimum:", df["credit_score"].min())
print("maximum:", df["credit_score"].max())

dtype: float64
missing: 70
minimum: 501.0
maximum: 999.0


In [16]:
invalid_score = df["credit_score"].notna() & ~df["credit_score"].between(500, 850)

print("Invalid score rows:", invalid_score.sum())
display(df.loc[invalid_score, ["applicant_id", "credit_score"]].head(10))

Invalid score rows: 13


,applicant_id,credit_score
124,APP01523,999.0
149,APP00390,999.0
324,APP01861,999.0
535,APP00662,999.0
651,APP01022,999.0
735,APP01502,999.0
736,APP01571,999.0
740,APP00120,999.0
888,APP01828,999.0
1151,APP00775,999.0


In [17]:
df.loc[invalid_score, "credit_score"] = np.nan
print("Missing scores after correction:", df["credit_score"].isna().sum())

Missing scores after correction: 83


In [18]:
print("Age <= 0:", (df["age"] <= 0).sum())
print("Monthly income <= 0:", (df["monthly_income"] <= 0).sum())
print("Requested amount <= 0:", (df["requested_amount"] <= 0).sum())
print("Current EMI < 0:", (df["current_monthly_emi"] < 0).sum())
print("Approved amount <= 0:", (df["approved_loan_amount"] <= 0).sum())

Age <= 0: 11
Monthly income <= 0: 10
Requested amount <= 0: 14
Current EMI < 0: 10
Approved amount <= 0: 11


In [19]:
df.loc[df["age"] <= 0, "age"] = np.nan
df.loc[df["monthly_income"] <= 0, "monthly_income"] = np.nan
df.loc[df["requested_amount"] <= 0, "requested_amount"] = np.nan
df.loc[df["current_monthly_emi"] < 0, "current_monthly_emi"] = np.nan
df.loc[df["approved_loan_amount"] <= 0, "approved_loan_amount"] = np.nan

print(df.isna().sum())

applicant_id             0
age                     11
monthly_income          67
credit_score            83
existing_loans           0
employment_type         57
loan_tenure_months       0
requested_amount        14
current_monthly_emi     57
approved_loan_amount    35
record_updated_at        0
dtype: int64


In [20]:
print(df["employment_type"].value_counts(dropna=False))

employment_type
Salaried          1082
Self-employed      444
Business           218
NaN                 57
salaried            42
 SALARIED           42
Salary              37
self employed       21
self-employed       19
BUSINESS            15
SELF-EMPLOYED       12
business            11
Name: count, dtype: int64


In [21]:
df["employment_type"] = (
    df["employment_type"]
      .astype("string")
      .str.strip()
      .str.lower()
      .replace({
          "salary": "salaried",
          "self employed": "self-employed"
      })
)

print(df["employment_type"].value_counts(dropna=False))

employment_type
salaried         1203
self-employed     496
business          244
<NA>               57
Name: count, dtype: Int64


In [22]:
from sklearn.model_selection import train_test_split

model_df = df.dropna(subset=["approved_loan_amount"]).copy()

feature_cols = [
    "age",
    "monthly_income",
    "credit_score",
    "existing_loans",
    "employment_type",
    "loan_tenure_months",
    "requested_amount",
    "current_monthly_emi"
]

X = model_df[feature_cols]
y = model_df["approved_loan_amount"].astype(float)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print("Usable rows:", len(model_df))
print("Train:", X_train.shape, y_train.shape)
print("Test:", X_test.shape, y_test.shape)

Usable rows: 1965
Train: (1572, 8) (1572,)
Test: (393, 8) (393,)


In [23]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

numeric_cols = [
    "age", "monthly_income", "credit_score", "existing_loans",
    "loan_tenure_months", "requested_amount", "current_monthly_emi"
]
categorical_cols = ["employment_type"]

numeric_steps = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_steps = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_steps, numeric_cols),
    ("categorical", categorical_steps, categorical_cols)
])

In [24]:
from sklearn.linear_model import LinearRegression

model = Pipeline([
    ("preprocessing", preprocessor),
    ("regression", LinearRegression())
])

model.fit(X_train, y_train)

train_pred = model.predict(X_train)
test_pred = model.predict(X_test)

print("First 5 actual amounts:", y_test.iloc[:5].to_numpy())
print("First 5 predictions:", test_pred[:5].round(0))

TypeError: boolean value of NA is ambiguous

In [25]:
for part in (X_train, X_test):
    for col in numeric_cols:
        part[col] = part[col].astype("float64")

    part["employment_type"] = (
        part["employment_type"]
        .astype("object")
        .where(part["employment_type"].notna(), np.nan)
    )

print(X_train.dtypes)
print(X_train.isna().sum())

age                    float64
monthly_income         float64
credit_score           float64
existing_loans         float64
employment_type         object
loan_tenure_months     float64
requested_amount       float64
current_monthly_emi    float64
dtype: object
age                     9
monthly_income         51
credit_score           66
existing_loans          0
employment_type        44
loan_tenure_months      0
requested_amount       10
current_monthly_emi    44
dtype: int64


In [26]:
model.fit(X_train, y_train)

train_pred = model.predict(X_train)
test_pred = model.predict(X_test)

print("First 5 actual amounts:", y_test.iloc[:5].to_numpy())
print("First 5 predictions:", test_pred[:5].round(0))

First 5 actual amounts: [332000. 154000. 423000. 626000. 523000.]
First 5 predictions: [263681. 151825. 538981. 553712. 463293.]


In [27]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

for name, actual, predicted in [
    ("Train", y_train, train_pred),
    ("Test", y_test, test_pred)
]:
    mae = mean_absolute_error(actual, predicted)
    rmse = np.sqrt(mean_squared_error(actual, predicted))
    r2 = r2_score(actual, predicted)

    print(f"{name}: MAE = ₹{mae:,.0f}, RMSE = ₹{rmse:,.0f}, R² = {r2:.3f}")

Train: MAE = ₹74,716, RMSE = ₹98,109, R² = 0.864
Test: MAE = ₹80,594, RMSE = ₹118,716, R² = 0.788


In [28]:
baseline_pred = np.full(len(y_test), y_train.mean())

print("Baseline MAE:", round(mean_absolute_error(y_test, baseline_pred)))
print("Model MAE:", round(mean_absolute_error(y_test, test_pred)))

Baseline MAE: 195634
Model MAE: 80594


In [29]:
results = X_test.copy()
results["actual"] = y_test
results["predicted"] = test_pred
results["error"] = results["actual"] - results["predicted"]
results["absolute_error"] = results["error"].abs()

display(
    results.sort_values("absolute_error", ascending=False)
           [["requested_amount", "monthly_income", "credit_score",
             "actual", "predicted", "error", "absolute_error"]]
           .head(10)
)

,requested_amount,monthly_income,credit_score,actual,predicted,error,absolute_error
73,3000000.0,NaN,635.0,1482000.0,4.458618e+05,1.036138e+06,1.036138e+06
1306,3000000.0,NaN,582.0,1101000.0,2.920428e+05,8.089572e+05,8.089572e+05
1269,1727000.0,NaN,626.0,761000.0,3.899963e+05,3.710037e+05,3.710037e+05
1764,1998000.0,196982.0,690.0,1682000.0,1.316050e+06,3.659502e+05,3.659502e+05
598,1044000.0,NaN,830.0,837000.0,4.711957e+05,3.658043e+05,3.658043e+05
1576,434000.0,NaN,839.0,179000.0,5.290850e+05,-3.500850e+05,3.500850e+05
259,1995000.0,NaN,734.0,850000.0,5.209234e+05,3.290766e+05,3.290766e+05
1562,850000.0,NaN,810.0,215000.0,4.634178e+05,-2.484178e+05,2.484178e+05
935,438000.0,36051.0,620.0,399000.0,1.581653e+05,2.408347e+05,2.408347e+05
1597,592000.0,NaN,805.0,265000.0,5.024712e+05,-2.374712e+05,2.374712e+05


In [30]:
print(
    results.groupby(results["monthly_income"].isna())["absolute_error"]
           .agg(["count", "mean", "median"])
)

                count           mean         median
monthly_income                                     
False             379   72358.301223   66084.045588
True               14  303559.106273  242944.544653
